# Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import logging
import os
import sys
import pandas as pd

# enforce more deterministic behavior in cuBLAS operations.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
# select a GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

sys.path.append("..")

from processor.core.interaction_conductor.llm_conductor import LLMConductor
from processor.core.ir_system.ir_data_model import RetrieverType
from processor.utils.logger import setup_logger
from processor.core.ir_system.ir_data_model import AbstractDocument
from processor.core.ir_system.ir_data_model import Table, TableContext
from processor.core.ir_system.ir_data_model import Knowledge
from processor.model.interface.model_factory import get_embed_model, get_llm
from processor.model.llm_message import Role
from processor.model.option import LLMOption
from processor.utils.json_processor import parse_json

from logging import Logger
from pandas import DataFrame

logger = setup_logger(
    name="processor_logger",
    log_path=os.path.join(".", "log"),
    level=logging.INFO,
    max_bytes=10_000_000,
    backup_count=5,
)

# IR System

In [ ]:
# from processor.core.ir_system.lm_interface import LMInterface


# lm_interface = LMInterface({
#     "llm": get_llm("gpt-4o-mini")("gpt-4o-mini"),
#     "embed_model": get_embed_model()("model/weight/bge-base"),
# },
# logger)

In [ ]:
# output = lm_interface.retrieve(
#     RetrieverType.PNEUMA, "I need some shipping data.", ["buysite"], 3
# )

In [ ]:
# for i in output:
#     print(i.doc_id)

In [ ]:
# output2 = lm_interface.re_retrieve_with_feedback(
#     RetrieverType.PNEUMA, "Advanced shipping notice is relevant, but I need shipment numbers associated with them.", output, 3, ["buysite"]
# )

In [ ]:
# for i in output2:
#     print(i.doc_id)

# LLMConductor

In [18]:
class Interaction:
    """
    Basically keeps track of every call to the process_input() function of LLMConductor
    """
    def __init__(self, human_input: str, llm_response: str) -> None:
        self.human_input = human_input
        self.llm_response = llm_response

    def __str__(self) -> str:
        return f"""{{"human input": {self.human_input}, "llm response": {self.llm_response}}}"""

class InformationNeedState:
    """
    Represents user's information need as a set of target schemas and SQLs to be executed over them.
    For example, if the user needs to know about the work addresses of faculty members, the target schemas
    may be ["name", "work address"], where name represents the names of the members, and work address represents
    the corresponding work address of each of them. After materialized by Materializer Engine, the SQLs can be
    executed sequentially over the materialized tables, and the outcome is useful to answer user's needs.
    """
    def __init__(self) -> None:
        self.target_schemas: dict[str, DataFrame] = dict()
        self.is_target_schemas_materialized = False
        self.column_descriptions: dict[str, dict[str, str]] = dict()
        self.sqls: list[str] = []

    def __str__(self) -> str:
        target_schemas_repr = ""
        for schema_id in self.target_schemas:
            table = self.target_schemas[schema_id]
            target_schemas_repr += f"\n- Table {schema_id}:\ncol: {" | ".join(list(table.columns))}"
            if len(table) > 0:
                # Sample 5 rows to represent the table
                sample_rows = table.sample(min(5, len(table)), random_state=42)
                sample_row_idx = 1
                for _, data in sample_rows.iterrows():
                    str_data = [str(i) for i in data]
                    target_schemas_repr += (
                        f"\n- sample row {sample_row_idx}: {" | ".join(str_data)}"
                    )
                    sample_row_idx += 1
            target_schemas_repr += "\n"
        return f"""Target schemas:
{target_schemas_repr.strip()}

Column descriptions of target schemas:
{self.column_descriptions}

SQLs to be run sequentially over the target schemas:
{self.sqls}"""

In [ ]:
from processor.model.llm_message import LLMMessage


class ICPromptFactory:
    def get_sys_prompt(self, iteration_limit: int) -> str:
      return f"""You are the orchestrator of a system that helps users fulfill complex information needs using structured data and interactive dialogue. Your job is to **collaboratively understand the user's goals**, which may be vague (i.e., you have to guide the user using the available data), and retrieve and structure relevant data, and iteratively build towards an accurate result.

### CONTEXT
- You are in an ongoing conversation with the user.
- There are **{iteration_limit}** allowed stes in this turn.
- You maintain an internal **state** that includes:
  - Target schemas (structured representation of what the user wants)
  - Column descriptions of the target schemas
  - SQL queries over those schemas
- The user may not fully know what they need—**elicit, don't assume**.
- In some cases, a "force response anyway" instruction may be given, even if you're unsure how to proceed. In that case, **respond with your best-effort reasoning**, using the current state and available context. Reflect openly.

### CORE BEHAVIORS

1. **Clarify Intent**:
   - Ask minimal, concrete questions.
   - Never guess. Always verify unclear terms, fields, or goals.
2. **Use the IR System** (only when needed) to fetch relevant data.
3. **Interpret IR Results** to decide what data is available.
4. **Propose or update Target Schemas** aligned with:
   - What the user asked,
   - What the IR system returned.
5. **Confirm changes** with the user before materializing or querying.
6. **Respect the dependencies of the tools**: Materializer Engine requires that you define the target schemas and SQLs beforehand, and SQL Engine requires the target schemas to already be materialized.
7. **Stop and explain** your reasoning frequently.
8. **Respond to the user** when you have intermediate results, uncertainties, or clarifications to share.

### INTENT TYPES
At each step, choose one of:
- `communicate_with_user`: Explain progress (summary of all actions that you have taken in this turn) or ask for feedback.
- `internal_reasoning`: Reflect out loud (for yourself only).
- `tool_call`: Call a system tool with appropriate arguments.

### TOOLS

- **IR System**
  - Retrieves relevant documents or tables.
  - Args: `{{"prompt": "<your query>"}}`
  - Call only if:
    - No IR results exist,
    - Or existing ones are clearly irrelevant.
  - Prompts to the IR system may already encode the most optimal phrasing—**avoid overthinking or rephrasing when it's already clear**.
  - **Avoid repeated IR calls with similar prompts.**

- **State Manipulation**
  - Updates your understanding of target schemas and SQLs.
  - Args:
    {{
      "new_target_schemas": {{
        "<schema_id>": {{
          "<column>": "<description>"
        }}
      }},
      "new_sqls": ["..."]
    }}
  - Never use real database names. Refer only to schema IDs.

- **Materializer Engine**
  - Populates the target schemas with actual data.
  - Args: `""`
  - Only call after schemas are finalized (communicated and agreed with the user).

- **SQL Engine**
  - Runs SQLs over materialized schemas.
  - Args: `""`
  - Only call if the target schemas have been materialized and the user agrees that they represent what the user needs.

### STRATEGY

- Do not rush. You have multiple steps per turn, and multiple turns per session.
- If unsure, **ask** rather than guess.
- If results are confusing or unexpected, stop and **explain your thoughts**.
- If the user seems to want a quick answer (e.g., "what's 3x5?"), feel free to respond directly. Or for a general knowledge question, you may know the answer, but clearly say it is according to the best of your knowledge.
- Be frugal with tool use. Reason before you call.
- Assume the prompt you receive reflects the **most optimized form already**, especially in forced response or fallback cases (e.g., IR prompt is pre-designed to be optimal).
- If asked to respond anyway, **think aloud**, even if partial.

### GOALS
- Build shared understanding with the user.
- Prioritize transparency over speed.
- Make steady progress by iteratively refining your state.
- Only finalize answers when confident they meet the user's intent."""

    def get_env_state_prompt(
        self,
        curr_iteration: int,
        max_iteration: int,
        info_need_state: InformationNeedState,
        interaction_history: list[Interaction],
        human_input: str,
    ) -> str:
        return f"""Relevant information for the current iteration step (iteration {curr_iteration} out of {max_iteration}):

INFORMATION NEED STATE:
{info_need_state}

CURRENT HUMAN INPUT:
{human_input}

INTERACTIONS HISTORY (HUMAN INPUT-YOUR HUMAN-FACING RESPONSE PAIRS):
{self.__convert_interactions_to_str(interaction_history)}

Please output your decision for this step in the following format:
{{
    "intent": "communicate_with_user" | "internal_reasoning" | "tool_call",
    "message": null | "<string>",
    "tool": null | "IR System" | "Materializer Engine" | "State Manipulation" | "SQL Engine",
    "args": null | { ... }
}}"""
    
    def get_direct_response_anyway_prompt(self) -> str:
        return """You have reached the iteration limit for this step. Please summarize the actions that you have done.
You are essentially asked to produce a `communicate_with_user` response but without the JSON format requirements. Simply output the summary."""   
    
    def __convert_interactions_to_str(self, interactions: list[Interaction]) -> str:
        interaction_repr = ""
        for interaction in interactions:
            interaction_repr += f"- {interaction}\n"
        interaction_repr = interaction_repr.strip()
        return interaction_repr

In [ ]:
ITERATION_LIMIT = 3
PAST_INTERACTIONS_LIMIT = 5


class LLMConductor:
    def __init__(self, llm_path: str, embed_path: str, logger: Logger) -> None:
        self.llm = get_llm(llm_path)(llm_path)
        self.embed_model = get_embed_model()(embed_path)
        self.logger = logger

        self.info_need_state = InformationNeedState()
        self.interaction_history: list[Interaction] = []

        self.prompt_factory = ICPromptFactory()

    def process_input(self, human_input: str) -> str:
        num_iteration = 0
        user_facing_response = ""
        is_user_facing_response = False
        llm_messages = [
            LLMMessage(
                role=Role.SYSTEM.value, content=self.prompt_factory.get_sys_prompt(ITERATION_LIMIT)
            )
        ]
        while not is_user_facing_response and num_iteration < ITERATION_LIMIT:
            num_iteration += 1
            llm_messages.append(
                LLMMessage(
                    role=Role.USER.value,
                    content=self.prompt_factory.get_env_state_prompt(
                        num_iteration,
                        ITERATION_LIMIT,
                        self.info_need_state,
                        self.interaction_history,
                        human_input,
                    ),
                )
            )

            llm_output = self.llm.chat(llm_messages, LLMOption(json_mode=True))
            llm_messages.append(
                LLMMessage(role=Role.ASSISTANT.value, content=llm_output)
            )
            """Format of action:
            {
                "intent": "communicate_with_user" | "internal_reasoning" | "tool_call",
                "message": null | "<string>",
                "tool": null | "IR System" | "Materializer Engine" | "State Manipulation" | "SQL Engine",
                "args": null | { ... }
            }
            """
            action = parse_json(llm_output)
            intent: str = action["intent"]
            action_message: None | str = action["message"]
            tool: None | str = action["tool"]
            args: None | dict = action["args"]

            if intent == "communicate_with_user" and isinstance(action_message, str):
                self.interaction_history.append(
                    Interaction(human_input, action_message)
                )
                user_facing_response = action_message
                is_user_facing_response = True
            elif intent == "internal_reasoning" and isinstance(action_message, str):
                llm_messages.append(
                    LLMMessage(
                        role=Role.USER.value,
                        content=f"You did some internal reasoning: {action_message}",
                    )
                )
            elif intent == "tool_call" and tool is not None and args is not None:
                tool_outcome = self.__execute_tool(tool, args)
                llm_messages.append(
                    LLMMessage(role=Role.USER.value, content=tool_outcome)
                )

        if not is_user_facing_response:
            llm_messages.append(
                LLMMessage(
                    role=Role.SYSTEM.value,
                    content=self.prompt_factory.get_direct_response_anyway_prompt(),
                )
            )
            user_facing_response = self.llm.chat(llm_messages)
            self.interaction_history.append(
                Interaction(human_input, user_facing_response)
            )
        return user_facing_response

    def __execute_tool(self, tool: str, args: str | dict) -> str:
        return ""  # To be implemented


# llm_path = "gpt-4o-mini"
llm_path = "model/weight/qwen3-1_7b"
embed_model_path = "model/weight/bge-base"
llm_conductor = LLMConductor(llm_path, embed_model_path, logger)
llm_conductor.process_input("I want to find the nearest store to me!")